# Nettoyage des données

Ce notebook applique les décisions du contrôle qualité (`01_data-quality-check.ipynb`) et écrit des tables propres dans `data_clean/`, prêtes pour l'analyse. Les données d'origine dans `data/` ne sont jamais modifiées.

In [1]:
import os, duckdb
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
os.makedirs("data_clean", exist_ok=True)
con = duckdb.connect()
con.execute("SET TimeZone = 'UTC'")

## 1. Visiteurs

### Genre
Le même genre est écrit de plusieurs façons (F, Female, female). Je les regroupe pour ne pas fausser les comptages, et je remplace les vides par `unknown` pour garder ces visiteurs dans l'analyse.

In [2]:
query = """SELECT DISTINCT gender FROM 'data/visitors.parquet'"""
con.sql(query).df()

,gender
0,other
1,prefer_not_to_say
2,F
3,Female
4,M
5,Male
6,female
7,NaN
8,male


In [3]:
query = """
    SELECT gender AS before,
        CASE
            WHEN gender IN ('female', 'F', 'Female') THEN 'female'
            WHEN gender IN ('male', 'M', 'Male')     THEN 'male'
            WHEN gender IS NULL                      THEN 'unknown'
            ELSE gender
        END AS after,
        COUNT(*) AS n
    FROM 'data/visitors.parquet'
    GROUP BY ALL
    ORDER BY after
"""
con.sql(query).df()

,before,after,n
0,female,female,9301
1,F,female,283
2,Female,female,219
3,M,male,245
4,Male,male,246
5,male,male,8257
6,other,other,284
7,prefer_not_to_say,prefer_not_to_say,512
8,NaN,unknown,653


### Région
Certaines régions sont écrites en minuscules ou sans accents (« bretagne », « Ile de France »). Je les ramène au nom officiel, et je remplace les vides par `unknown`.

In [4]:
query = """
    SELECT region AS before,
        CASE
            WHEN region IS NULL                         THEN 'unknown'
            WHEN lower(region) = 'ile de france'        THEN 'Île-de-France'
            WHEN lower(region) = 'auvergne rhone alpes' THEN 'Auvergne-Rhône-Alpes'
            WHEN region = 'pays de la loire'            THEN 'Pays de la Loire'
            WHEN region = 'bretagne'                    THEN 'Bretagne'
            WHEN region = 'occitanie'                   THEN 'Occitanie'
            WHEN region = 'nouvelle-aquitaine'          THEN 'Nouvelle-Aquitaine'
            WHEN region = 'grand est'                   THEN 'Grand Est'
            WHEN region = 'hauts-de-france'             THEN 'Hauts-de-France'
            WHEN region = 'provence-alpes-côte d''azur' THEN 'Provence-Alpes-Côte d''Azur'
            ELSE region
        END AS after,
        COUNT(*) AS n
    FROM 'data/visitors.parquet'
    GROUP BY ALL
    ORDER BY after
"""
con.sql(query).df()

,before,after,n
0,Auvergne Rhone Alpes,Auvergne-Rhône-Alpes,103
1,Auvergne-Rhône-Alpes,Auvergne-Rhône-Alpes,2080
2,bretagne,Bretagne,105
3,Bretagne,Bretagne,2014
4,Grand Est,Grand Est,2022
5,grand est,Grand Est,99
6,Hauts-de-France,Hauts-de-France,2132
7,hauts-de-france,Hauts-de-France,88
8,Nouvelle-Aquitaine,Nouvelle-Aquitaine,2014
9,nouvelle-aquitaine,Nouvelle-Aquitaine,99


18 libellés ramenés aux 9 régions officielles ; 804 vides passés en `unknown`.

### Âge
L'année de naissance seule est peu lisible. J'ajoute une tranche d'âge (âge calculé en 2026). Les 42 mineurs gardent leur propre tranche `under_18`, et les vides passent en `unknown`.

In [5]:
query = """
    SELECT
        CASE
            WHEN birth_year IS NULL     THEN 'unknown'
            WHEN 2026 - birth_year < 18 THEN 'under_18'
            WHEN 2026 - birth_year < 25 THEN '18-24'
            WHEN 2026 - birth_year < 35 THEN '25-34'
            WHEN 2026 - birth_year < 45 THEN '35-44'
            WHEN 2026 - birth_year < 55 THEN '45-54'
            WHEN 2026 - birth_year < 65 THEN '55-64'
            ELSE '65+'
        END AS age_band,
        MIN(2026 - birth_year) AS age_min,
        MAX(2026 - birth_year) AS age_max,
        COUNT(*) AS n,
        100.0 * COUNT(*) / (SELECT COUNT(*) FROM 'data/visitors.parquet')  AS percentage_of_total
    FROM 'data/visitors.parquet'
    GROUP BY ALL
    ORDER BY age_min
"""
con.sql(query).df()

,age_band,age_min,age_max,n,percentage_of_total
0,under_18,16,17,42,0.210
1,18-24,18,24,2095,10.475
2,25-34,25,34,3207,16.035
3,35-44,35,44,4583,22.915
4,45-54,45,54,4393,21.965
5,55-64,55,64,2920,14.600
6,65+,65,97,1999,9.995
7,unknown,<NA>,<NA>,761,3.805


7 tranches d'âge sans chevauchement (16 à 97 ans) ; 761 années manquantes → `unknown`.

### Écriture de la table
J'applique les trois transformations (genre, région, tranche d'âge) et j'écris le résultat dans `data_clean/visitors.parquet`.

In [6]:
query = """
COPY (
    SELECT visitor_id, birth_year, first_seen_at,
        CASE
            WHEN gender IN ('female', 'F', 'Female') THEN 'female'
            WHEN gender IN ('male', 'M', 'Male')     THEN 'male'
            WHEN gender IS NULL                      THEN 'unknown'
            ELSE gender
        END AS gender,
        CASE
            WHEN region IS NULL                         THEN 'unknown'
            WHEN lower(region) = 'ile de france'        THEN 'Île-de-France'
            WHEN lower(region) = 'auvergne rhone alpes' THEN 'Auvergne-Rhône-Alpes'
            WHEN region = 'pays de la loire'            THEN 'Pays de la Loire'
            WHEN region = 'bretagne'                    THEN 'Bretagne'
            WHEN region = 'occitanie'                   THEN 'Occitanie'
            WHEN region = 'nouvelle-aquitaine'          THEN 'Nouvelle-Aquitaine'
            WHEN region = 'grand est'                   THEN 'Grand Est'
            WHEN region = 'hauts-de-france'             THEN 'Hauts-de-France'
            WHEN region = 'provence-alpes-côte d''azur' THEN 'Provence-Alpes-Côte d''Azur'
            ELSE region
        END AS region,
        CASE
            WHEN birth_year IS NULL     THEN 'unknown'
            WHEN 2026 - birth_year < 18 THEN 'under_18'
            WHEN 2026 - birth_year < 25 THEN '18-24'
            WHEN 2026 - birth_year < 35 THEN '25-34'
            WHEN 2026 - birth_year < 45 THEN '35-44'
            WHEN 2026 - birth_year < 55 THEN '45-54'
            WHEN 2026 - birth_year < 65 THEN '55-64'
            ELSE '65+'
        END AS age_band
    FROM 'data/visitors.parquet'
) TO 'data_clean/visitors.parquet' (FORMAT parquet)
"""
con.execute(query)

con.sql("SELECT COUNT(*) AS n FROM 'data_clean/visitors.parquet'").df()

,n
0,20000


In [7]:
con.sql("SELECT * FROM 'data_clean/visitors.parquet' LIMIT 5").df()

,visitor_id,birth_year,first_seen_at,gender,region,age_band
0,VIS_00000001,2008,2026-08-02 04:29:33.074952+00:00,male,Bretagne,18-24
1,VIS_00000002,1946,2026-07-25 11:59:57.752213+00:00,female,Hauts-de-France,65+
2,VIS_00000003,<NA>,2026-06-22 12:37:47.114315+00:00,female,Pays de la Loire,unknown
3,VIS_00000004,1989,2026-07-12 02:29:05.408274+00:00,other,Provence-Alpes-Côte d'Azur,35-44
4,VIS_00000005,1989,2026-07-10 05:33:03.390070+00:00,female,Hauts-de-France,35-44


## 2. Sessions

### Source d'acquisition
Une même source a plusieurs noms. Je les regroupe par type de canal : Google/google → `google_organic`, facebook/instagram → `social` (la majorité du trafic social n'indique pas le réseau), ChatGPT/AI Assistant → `ai_assistant`.

In [8]:
con.sql("""
SELECT DISTINCT acquisition_source FROM 'data/sessions.parquet'

""")

┌────────────────────┐
│ acquisition_source │
│      varchar       │
├────────────────────┤
│ google_ads         │
│ referral           │
│ other              │
│ chatgpt            │
│ facebook           │
│ google             │
│ google_organic     │
│ direct             │
│ Google             │
│ instagram          │
│ ChatGPT            │
│ AI Assistant       │
│ social             │
│ email              │
└────────────────────┘
       14 rows      

In [9]:
query = """
    SELECT acquisition_source AS before,
        CASE
            WHEN acquisition_source IN ('Google', 'google')                   THEN 'google_organic'
            WHEN acquisition_source IN ('chatgpt', 'ChatGPT', 'AI Assistant') THEN 'ai_assistant'
            WHEN acquisition_source IN ('facebook', 'instagram')              THEN 'social'
            ELSE acquisition_source
        END AS after,
        COUNT(*) AS n
    FROM 'data/sessions.parquet'
    GROUP BY ALL
    ORDER BY after
"""
con.sql(query).df()

,before,after,n
0,AI Assistant,ai_assistant,165
1,ChatGPT,ai_assistant,163
2,chatgpt,ai_assistant,2401
3,direct,direct,1617
4,email,email,1707
5,google_ads,google_ads,6375
6,google_organic,google_organic,6590
7,Google,google_organic,450
8,google,google_organic,450
9,other,other,674


14 libellés ramenés à 8 canaux.

### Campagne
Une campagne vide est normale pour les sources organiques (google_organic, referral…). Je la remplace par `no_campaign` pour que ces sessions restent visibles dans les regroupements.

In [10]:
query = """
    SELECT campaign_name AS before,
           COALESCE(campaign_name, 'no_campaign') AS after,
           COUNT(*) AS n
    FROM 'data/sessions.parquet'
    GROUP BY ALL
    ORDER BY n DESC
"""
con.sql(query).df()

,before,after,n
0,NaN,no_campaign,14081
1,diabetes_awareness_general,diabetes_awareness_general,3422
2,diabetes_social_awareness,diabetes_social_awareness,3345
3,diabetes_45plus,diabetes_45plus,3337
4,diabetes_retargeting,diabetes_retargeting,1815


14 081 campagnes vides → `no_campaign` 

In [11]:
query = """
    SELECT campaign_name, is_returning_visitor, COUNT(*) AS n
    FROM 'data/sessions.parquet'
    WHERE acquisition_source = 'direct' AND campaign_name IS NOT NULL
    GROUP BY ALL
"""
con.sql(query).df()

,campaign_name,is_returning_visitor,n
0,diabetes_retargeting,True,219


Les 219 sessions `direct` avec une campagne sont toutes des visiteurs revenus via `diabetes_retargeting` : la source a probablement été mal attribuée. Je les garde en `direct` 

In [12]:
con.sql("""
SELECT DISTINCT landing_page FROM 'data/sessions.parquet'   
""")

┌────────────────────────────┐
│        landing_page        │
│          varchar           │
├────────────────────────────┤
│ /health/diabetes-awareness │
└────────────────────────────┘

### Écriture de la table
J'applique les deux transformations (source et campagne), je retire `landing_page` (même valeur sur toutes les lignes) et j'écris le résultat dans `data_clean/sessions.parquet`.

In [13]:
query = """
COPY (
    SELECT session_id, visitor_id, session_started_at, device_type, is_returning_visitor,
        CASE
            WHEN acquisition_source IN ('Google', 'google')                   THEN 'google_organic'
            WHEN acquisition_source IN ('chatgpt', 'ChatGPT', 'AI Assistant') THEN 'ai_assistant'
            WHEN acquisition_source IN ('facebook', 'instagram')              THEN 'social'
            ELSE acquisition_source
        END AS acquisition_source,
        COALESCE(campaign_name, 'no_campaign') AS campaign_name
    FROM 'data/sessions.parquet'
) TO 'data_clean/sessions.parquet' (FORMAT parquet)
"""
con.execute(query)

con.sql("SELECT COUNT(*) AS n FROM 'data_clean/sessions.parquet'").df()

,n
0,26000


## 3. Réponses

### Réponses modifiées
Selon `01_data-quality-check.ipynb`, 697 questions ont reçu deux réponses dans la même session, c'est-à-dire changement d'avis. Je garde la dernière : c'est elle qui détermine le résultat.

Exemple : la première question répondue deux fois.

In [14]:
query = """
    WITH premier_doublon AS (
        SELECT session_id, question_number
        FROM 'data/questionnaire_answers.parquet'
        GROUP BY session_id, question_number
        HAVING COUNT(*) > 1
        ORDER BY session_id
        LIMIT 1
    )
    SELECT a.session_id, a.question_number, a.answer_value, a.answered_at,
           ROW_NUMBER() OVER (PARTITION BY a.session_id, a.question_number
                              ORDER BY a.answered_at DESC) AS rang
    FROM 'data/questionnaire_answers.parquet' a
    JOIN premier_doublon USING (session_id, question_number)
"""
con.sql(query).df()

,session_id,question_number,answer_value,answered_at,rang
0,SES_00000003,4,3_to_4_days_per_week,2026-06-01 00:12:37.189775+00:00,1
1,SES_00000003,4,1_to_2_days_per_week,2026-06-01 00:12:19.217151+00:00,2


### Écriture de la table
Je garde la réponse de rang 1 (la plus récente) pour chaque question de chaque session.

In [15]:
query = """
COPY (
    SELECT *
    FROM 'data/questionnaire_answers.parquet'
    QUALIFY ROW_NUMBER() OVER (PARTITION BY session_id, question_number
                               ORDER BY answered_at DESC) = 1
) TO 'data_clean/questionnaire_answers.parquet' (FORMAT parquet)
"""
con.execute(query)

con.sql("SELECT COUNT(*) AS n FROM 'data_clean/questionnaire_answers.parquet'").df()

,n
0,67763


697 anciennes réponses retirées : 67 763 (68 460 − 697) réponses, une par question et par session.

## 4. Événements

### Événements en double
Selon `01_data-quality-check.ipynb`, 1 043 `question_view` sont des doublons techniques et 697 `question_answer` sont des changements d'avis. Comme décidé, je garde la première vue et la dernière réponse, cohérente avec la table des réponses nettoyée.

Combien de doublons par type d'événement :

In [16]:
query = """
    SELECT event_type, COUNT(*) AS groupes_en_double
    FROM (
        SELECT session_id, event_type, question_number
        FROM 'data/questionnaire_events.parquet'
        GROUP BY ALL
        HAVING COUNT(*) > 1
    )
    GROUP BY event_type
"""
con.sql(query).df()

,event_type,groupes_en_double
0,question_answer,697
1,question_view,1043


### Écriture de la table

In [17]:
query = """
COPY (
    (SELECT *
     FROM 'data/questionnaire_events.parquet'
     WHERE event_type <> 'question_answer'
     QUALIFY ROW_NUMBER() OVER (PARTITION BY session_id, event_type, question_number
                                ORDER BY event_timestamp) = 1)
    UNION ALL
    (SELECT *
     FROM 'data/questionnaire_events.parquet'
     WHERE event_type = 'question_answer'
     QUALIFY ROW_NUMBER() OVER (PARTITION BY session_id, question_number
                                ORDER BY event_timestamp DESC) = 1)
) TO 'data_clean/questionnaire_events.parquet' (FORMAT parquet)
"""
con.execute(query)

con.sql("SELECT COUNT(*) AS n FROM 'data_clean/questionnaire_events.parquet'").df()

,n
0,194076


1 043 vues en double et 697 anciennes réponses retirées : 194 076 événements.

## 5. Tables sans correction
`questionnaire_outcomes` et `questionnaire_questions` n'ont aucun problème de qualité d'après `01_data-quality-check.ipynb`. Je les copie telles quelles pour que `data_clean/` contienne toutes les tables.

In [18]:
for t in ["questionnaire_outcomes", "questionnaire_questions"]:
    con.execute(f"COPY (SELECT * FROM 'data/{t}.parquet') TO 'data_clean/{t}.parquet' (FORMAT parquet)")

In [19]:
con.sql("SELECT COUNT(*) AS n FROM 'data_clean/questionnaire_outcomes.parquet'").df()

,n
0,11357


In [20]:
con.sql("SELECT COUNT(*) AS n FROM 'data_clean/questionnaire_questions.parquet'").df()

,n
0,5


## 6. Table d'analyse par session
Je réunis en une ligne par session tout ce qui la décrit : canal, campagne, profil du visiteur, étapes atteintes et résultat. Mes axes d'analyses peuvent ainsi partir de cette seule table.

In [21]:
query = """
COPY (
    WITH demarrees AS (
        SELECT DISTINCT session_id
        FROM 'data_clean/questionnaire_events.parquet'
        WHERE event_type = 'questionnaire_start'
    ),
    reponses AS (
        SELECT session_id, COUNT(*) AS nb_answers
        FROM 'data_clean/questionnaire_answers.parquet'
        GROUP BY session_id
    )
    SELECT
        s.session_id, s.visitor_id, s.session_started_at,
        date_trunc('week', s.session_started_at) AS week,
        s.acquisition_source, s.campaign_name, s.device_type, s.is_returning_visitor,
        v.gender, v.region, v.age_band,
        d.session_id IS NOT NULL                 AS started,
        COALESCE(r.nb_answers, 0)                AS nb_answers,
        o.session_id IS NOT NULL                 AS completed,
        o.outcome_category
    FROM 'data_clean/sessions.parquet' s
    JOIN      'data_clean/visitors.parquet'               v USING (visitor_id)
    LEFT JOIN demarrees                                   d USING (session_id)
    LEFT JOIN reponses                                    r USING (session_id)
    LEFT JOIN 'data_clean/questionnaire_outcomes.parquet' o USING (session_id)
) TO 'data_clean/sessions_analysis.parquet' (FORMAT parquet)
"""
con.execute(query)

In [22]:
con.sql("""
    SELECT COUNT(*)            AS sessions,
           SUM(started::INT)   AS demarrees,
           SUM(completed::INT) AS terminees,
           SUM(CASE WHEN nb_answers = 5 AND NOT completed THEN 1 ELSE 0 END) AS cinq_reponses_sans_resultat
    FROM 'data_clean/sessions_analysis.parquet'
""").df()

,sessions,demarrees,terminees,cinq_reponses_sans_resultat
0,26000,16774.0,11357.0,105.0


26 000 sessions, une ligne chacune ; les comptages (démarrées, terminées, 105 sans résultat) sont identiques au notebook 01.

## 7. Récapitulatif

| Table | Transformation | Justification  |
|---|---|---|
| visitors | Genre et région unifiés ; vides → `unknown` | Les variantes désignent la même valeur |
| visitors | Tranche d'âge (`under_18` … `65+`) | Âge = 2026 − année de naissance (± 1 an) ; mineurs gardés |
| sessions | Sources regroupées par canal ; campagne vide → `no_campaign` | AI Assistant n'est pas forcément ChatGPT ; `landing_page` retirée (valeur unique) |
| sessions | 219 `direct` avec campagne gardées | Attribution probablement perdue (retargeting) |
| answers | Dernière réponse gardée (−697) | Le résultat dépend de la dernière réponse |
| events | Première vue, dernière réponse (−1 740) | Doublons techniques et changements d'avis |
| sessions_analysis | Une ligne par session ; semaine = lundi | `nb_answers` = dernière question atteinte (aucune question sautée) ; 105 sessions à 5 réponses sans résultat → non terminées |